# PASO 1: Configuración y Carga de Datos

In [1]:
# ==============================================================================
# CELDA 1: CONFIGURACIÓN E IMPORTACIONES
# ==============================================================================
import pandas as pd
import numpy as np
import os
from google.colab import drive

# 1. Montar disco y forzar reconexión (Por si acaso)
drive.mount('/content/drive', force_remount=True)

# 2. Definir rutas
BASE_DIR = '/content/drive/MyDrive/CICLO_9/Integrador/Entrenamiento_GNN'
INPUT_FILE = os.path.join(BASE_DIR, 'Datos_Procesados', 'dataset_features_gnn.parquet')
OUTPUT_FILE = os.path.join(BASE_DIR, 'Datos_Procesados', 'tensor_panel_diario.parquet')

# 3. Cargar datos base
print("Cargando microdatos base...")
df_eventos = pd.read_parquet(INPUT_FILE)
df_eventos['fecha_delito'] = pd.to_datetime(df_eventos['fecha_delito'])
print(f"Total de eventos cargados: {len(df_eventos)}")

Mounted at /content/drive
Cargando microdatos base...
Total de eventos cargados: 573527


# PASO 2: Agrupación (Calculando la Densidad)

In [2]:
# ==============================================================================
# CELDA 2: CÁLCULO DE LA DENSIDAD CRIMINAL DIARIA
# ==============================================================================
print("Aplastando eventos en conteos diarios (Calculando variable Y)...")

# Contamos cuántas veces aparece un delito en la misma fecha y en el mismo nodo
df_agrupado = df_eventos.groupby(['fecha_delito', 'id_nodo']).size().reset_index(name='conteo_delitos')

print(f"Eventos agrupados exitosamente. Filas resultantes: {len(df_agrupado)}")

Aplastando eventos en conteos diarios (Calculando variable Y)...
Eventos agrupados exitosamente. Filas resultantes: 286802


# PASO 3: La Grilla de Tiempo (El Calendario Perfecto)

In [3]:
# ==============================================================================
# CELDA 3: CREACIÓN DEL CALENDARIO SIN HUECOS Y RELLENO DE CEROS
# ==============================================================================
# 1. Extraemos los límites de tiempo de tu dataset
fecha_min = df_eventos['fecha_delito'].min()
fecha_max = df_eventos['fecha_delito'].max()
nodos_unicos = range(400) # Tus 400 hotspots

print(f"Creando grilla de tiempo ininterrumpida desde {fecha_min.date()} hasta {fecha_max.date()}...")
rango_fechas = pd.date_range(start=fecha_min, end=fecha_max, freq='D')

# 2. Creamos todas las combinaciones posibles (Días x Nodos)
grilla = pd.MultiIndex.from_product(
    [rango_fechas, nodos_unicos],
    names=['fecha_delito', 'id_nodo']
).to_frame(index=False)

# 3. Cruzamos el calendario vacío con nuestros datos reales
df_panel = pd.merge(grilla, df_agrupado, on=['fecha_delito', 'id_nodo'], how='left')

# 4. Si un nodo no tuvo robos en un día específico, rellenamos con 0
df_panel['conteo_delitos'] = df_panel['conteo_delitos'].fillna(0).astype(int)

print("Calendario perfecto creado y ceros imputados.")

Creando grilla de tiempo ininterrumpida desde 2022-01-01 hasta 2026-03-31...
Calendario perfecto creado y ceros imputados.


# PASO 4: Ingeniería de Características Temporales


In [4]:
# ==============================================================================
# CELDA 4: RECALCULANDO EL TIEMPO CÍCLICO
# ==============================================================================
print("Calculando features temporales para el nuevo panel...")

# 1. Día de la semana y fin de semana
df_panel['dia_semana'] = df_panel['fecha_delito'].dt.weekday
df_panel['es_fin_semana'] = (df_panel['dia_semana'] >= 4).astype(int)

# 2. Mes cíclico (Seno y Coseno)
df_panel['mes'] = df_panel['fecha_delito'].dt.month
df_panel['mes_sin'] = np.sin(2 * np.pi * df_panel['mes'] / 12)
df_panel['mes_cos'] = np.cos(2 * np.pi * df_panel['mes'] / 12)

# Ordenamos cronológicamente para que PyTorch no se confunda al leer el historial
df_panel = df_panel.sort_values(['id_nodo', 'fecha_delito']).reset_index(drop=True)

print("Variables temporales añadidas al tensor.")

Calculando features temporales para el nuevo panel...
Variables temporales añadidas al tensor.


# PASO 5: Guardado y Verificación

In [5]:
# ==============================================================================
# CELDA 5: EXPORTACIÓN Y RESUMEN
# ==============================================================================
# Guardar el tensor definitivo
df_panel.to_parquet(OUTPUT_FILE, index=False)

print("\n=========================================")
print("--- RESUMEN DEL TENSOR ESPACIOTEMPORAL ---")
print(f"Total de registros (Días x Nodos): {len(df_panel)}")
print(f"Días totales analizados: {len(rango_fechas)}")
print(f"Guardado en: {OUTPUT_FILE}")
print("=========================================\n")

print("Vista previa de los primeros 5 días del Nodo 0:")
display(df_panel[df_panel['id_nodo'] == 0].head(5))


--- RESUMEN DEL TENSOR ESPACIOTEMPORAL ---
Total de registros (Días x Nodos): 620400
Días totales analizados: 1551
Guardado en: /content/drive/MyDrive/CICLO_9/Integrador/Entrenamiento_GNN/Datos_Procesados/tensor_panel_diario.parquet

Vista previa de los primeros 5 días del Nodo 0:


,fecha_delito,id_nodo,conteo_delitos,dia_semana,es_fin_semana,mes,mes_sin,mes_cos
0,2022-01-01,0,2,5,1,1,0.5,0.866025
1,2022-01-02,0,2,6,1,1,0.5,0.866025
2,2022-01-03,0,1,0,0,1,0.5,0.866025
3,2022-01-04,0,1,1,0,1,0.5,0.866025
4,2022-01-05,0,3,2,0,1,0.5,0.866025
